# 06 -- Baseline comparison: plain_pooled / no_fusion / hard_two_stage

Trains the three required baselines (config-driven, same `architecture` flag pattern) and produces a per-dataset macro-F1 comparison table against whichever primary MoE variant was trained in notebook 05 -- `hard_two_stage` is the literal 'obvious two-step approach' this project needs to beat.


## Setup

Run this cell first. It's the ONLY cell you should need to edit: change
`CONFIG_OVERRIDES` (a list of `--set key.path=value` style dotted overrides,
same syntax as `training.run`'s CLI) to narrow `data.active_datasets`,
switch `architecture`, point at a different Drive folder, etc.


In [ ]:
# ---- Single config cell: this is the only cell you should need to edit ----
IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import subprocess, os
    REPO_DIR = '/content/dataset_moe_nids'
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', 'https://github.com/selimsidan/dataset_moe_nids', REPO_DIR], check=True)
    %cd $REPO_DIR
    %pip install -q -r requirements.txt

CONFIG_PATH = 'config/default.yaml'

# Dotted --set overrides, same syntax as training.run's CLI. Narrow
# data.active_datasets to 2-3 datasets here for a fast iteration cycle --
# every downstream module (registry, harmonizer, expert-bank sizing,
# checkpoints) adapts automatically, no other code changes needed.
CONFIG_OVERRIDES = [
    # 'data.active_datasets=[NF-UNSW-NB15-v3,NF-BoT-IoT-v3]',
    # 'architecture=moe_dataset_soft',
    # 'training.device=cuda',
]

from training.config import load_config
config = load_config(CONFIG_PATH, CONFIG_OVERRIDES)
print('run_name:', config['run_name'])
print('architecture:', config['architecture'])
print('active_datasets:', config['data']['active_datasets'])
print('checkpoint_dir:', config['training']['checkpoint_dir'])


In [ ]:
from training.dataset import prepare_datasets
from training.baseline_train import train_hard_two_stage, train_no_fusion, train_plain_pooled
from training.stage_c_jointfinetune import build_model_from_checkpoints
from evaluation.metrics import evaluate_per_dataset, evaluate_predictions
from evaluation.report import per_dataset_comparison_table
import copy, torch

data = prepare_datasets(config)
device = torch.device(config['training'].get('device', 'cpu'))

results, per_dataset_results = {}, {}

moe_model = build_model_from_checkpoints(config, data, device)  # requires notebook 05 already run
moe_model.eval()
with torch.no_grad():
    preds = moe_model.predict(torch.from_numpy(data.test.features)).numpy()
results[config['architecture']] = evaluate_predictions(data.test.class_idx, preds, data.class_names)
per_dataset_results[config['architecture']] = evaluate_per_dataset(data.test.class_idx, preds, data.test.dataset_name, data.class_names)

for name, train_fn in [('plain_pooled', train_plain_pooled), ('no_fusion', train_no_fusion), ('hard_two_stage', train_hard_two_stage)]:
    baseline_cfg = copy.deepcopy(config)
    baseline_cfg['architecture'] = name
    model = train_fn(baseline_cfg, data)
    model.eval()
    with torch.no_grad():
        if name == 'no_fusion':
            y_true_l, y_pred_l, ds_l = [], [], []
            for ds_name in model.dataset_names:
                mask = data.test.dataset_name == ds_name
                if mask.sum() == 0:
                    continue
                p = model.predict(torch.from_numpy(data.test.features[mask]), ds_name).numpy()
                y_true_l.append(data.test.class_idx[mask]); y_pred_l.append(p); ds_l.append(data.test.dataset_name[mask])
            import numpy as np
            y_true, preds, ds_names = np.concatenate(y_true_l), np.concatenate(y_pred_l), np.concatenate(ds_l)
        else:
            preds = model.predict(torch.from_numpy(data.test.features)).numpy()
            y_true, ds_names = data.test.class_idx, data.test.dataset_name
    results[name] = evaluate_predictions(y_true, preds, data.class_names)
    per_dataset_results[name] = evaluate_per_dataset(y_true, preds, ds_names, data.class_names)

table = per_dataset_comparison_table(per_dataset_results)
table
